# ZGT Ground-Truth Mask Boxes to MedSAM Segmentation

This notebook is the segmentation-only version of the MedSAM mammography pipeline, adapted for the ZGT test data prepared in VersaMammo format.

Instead of using detector-predicted boxes, it extracts prompt boxes from the ground-truth mask for each test image. If one mask contains multiple disconnected lesions/components, the notebook creates one box per connected component, then runs MedSAM once per component box and unions the predicted masks.

Outputs:

- per-image MedSAM union masks
- per-prompt masks
- Dice scores against the ground-truth mask
- overlay visualizations with GT mask, predicted mask, and prompt boxes

In [ ]:
from pathlib import Path
import json
import sys
from typing import Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from skimage import io
from skimage.measure import label, regionprops
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

In [ ]:
# -------------------------
# Configuration
# -------------------------
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "pipelines and experiments":
    PROJECT_ROOT = PROJECT_ROOT.parent

MEDSAM_ROOT = PROJECT_ROOT / "MedSAM"
VERSAMAMMO_ROOT = PROJECT_ROOT / "VersaMammo"
DETECTION_METRICS_DIR = VERSAMAMMO_ROOT / "downstream" / "Detection" / "metrics"
DATA_ROOT = VERSAMAMMO_ROOT / "datapre" / "segdetdata"

DATASET_PREFIX = "ZGT_VersaMammo"
TEST_DATASET = None  # None = use the fold with best detection map_50 if available. Or set e.g. "ZGT_VersaMammo_fold4".
BEST_METRIC = "map_50"

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
MEDSAM_CHECKPOINT = MEDSAM_ROOT / "work_dir" / "MedSAM" / "medsam_vit_b.pth"
OUTPUT_ROOT = PROJECT_ROOT / "pipelines and experiments" / "medsam_from_zgt_gt_boxes"

MEDSAM_IMAGE_SIZE = (1024, 1024)
MIN_COMPONENT_AREA = 1
BBOX_PADDING = 0
MASK_THRESHOLD = 0.5
BATCH_SIZE = 1

print(f"Project root: {PROJECT_ROOT}")
print(f"CUDA available: {torch.cuda.is_available()} | device: {DEVICE}")

In [ ]:
# Import MedSAM and the mammography dataset utilities.
if str(MEDSAM_ROOT) not in sys.path:
    sys.path.insert(0, str(MEDSAM_ROOT))

utils_dir = MEDSAM_ROOT / "notebooks" / "segmentation"
if str(utils_dir) not in sys.path:
    sys.path.insert(0, str(utils_dir))

from segment_anything import sam_model_registry
from utils.utils_mammo_automatic import (
    MedSAMMammoDataset,
    ComposeSampleTransforms,
    EnsureImageChannels,
    medsam_inference_single,
)

In [ ]:
def load_json(path: Path):
    with open(path) as f:
        return json.load(f)


def choose_best_dataset(metrics_dir: Path, dataset_prefix: str, metric: str) -> str:
    candidates = []
    for metrics_path in sorted(metrics_dir.glob(f"{dataset_prefix}_fold*/test_metrics.json")):
        metrics = load_json(metrics_path)
        if metric in metrics:
            candidates.append((float(metrics[metric]), metrics_path.parent.name))
    if not candidates:
        # Fall back to fold0 if metrics are not present. The fixed Test split should be identical across folds.
        return f"{dataset_prefix}_fold0"
    candidates.sort(reverse=True)
    print("Available detection folds:")
    for value, fold_name in candidates:
        print(f"  {fold_name}: {metric}={value:.4f}")
    return candidates[0][1]


DATASET_NAME = TEST_DATASET or choose_best_dataset(DETECTION_METRICS_DIR, DATASET_PREFIX, BEST_METRIC)
TEST_ROOT = DATA_ROOT / DATASET_NAME / "Test"
OUT_DIR = OUTPUT_ROOT / DATASET_NAME
MASK_OUT_DIR = OUT_DIR / "pred_masks"
OVERLAY_OUT_DIR = OUT_DIR / "overlays"
METRICS_OUT = OUT_DIR / "medsam_gt_prompt_metrics.csv"
SUMMARY_OUT = OUT_DIR / "medsam_gt_prompt_summary.json"

print(f"Selected test dataset: {DATASET_NAME}")
print(f"Test root: {TEST_ROOT}")
print(f"Output root: {OUT_DIR}")

In [ ]:
class ZGTVersaMammoDataset(MedSAMMammoDataset):
    """Use MedSAMMammoDataset with VersaMammo-style `img.jpg` + `mask.png` cases."""

    def _collect_files(self, root, file_paths):
        if file_paths is not None:
            return [Path(p) for p in file_paths]
        root = Path(root)
        return sorted(root.glob("*/img.jpg"))

    def _load_dicom_image(self, path: Path) -> np.ndarray:
        # The parent class calls this method from __getitem__. For ZGT/VersaMammo
        # cases, the image is already a preprocessed image file rather than DICOM.
        img = np.array(Image.open(path).convert("L"), dtype=np.float32)
        if self.normalize:
            lo, hi = float(img.min()), float(img.max())
            img = (img - lo) / max(hi - lo, 1e-8)
        return img.astype(np.float32)

    def _image_to_annotation_path(self, image_path: Path) -> Optional[Path]:
        mask_path = image_path.parent / "mask.png"
        return mask_path if mask_path.exists() else None

In [ ]:
class ConnectedComponentBoxesFromMask:
    """Load a GT mask and return one bbox per connected component."""

    def __init__(self, min_area: int = 1, bbox_padding: int = 0, threshold: float = 0.0):
        self.min_area = int(min_area)
        self.bbox_padding = int(bbox_padding)
        self.threshold = float(threshold)

    def __call__(self, sample):
        annotation_path = sample.get("annotation_path")
        image = sample["image"]
        _, H, W = image.shape

        if annotation_path is None:
            mask = np.zeros((H, W), dtype=np.uint8)
        else:
            mask = np.array(Image.open(annotation_path).convert("L"))
            mask = (mask > self.threshold).astype(np.uint8)

        if mask.shape != (H, W):
            raise ValueError(f"Mask/image shape mismatch: mask={mask.shape}, image={(H, W)}, path={annotation_path}")

        cc = label(mask > 0, connectivity=2)
        boxes = []
        component_masks = []
        for region in regionprops(cc):
            if region.area < self.min_area:
                continue
            y_min, x_min, y_max_excl, x_max_excl = region.bbox
            x_min = max(0, x_min - self.bbox_padding)
            y_min = max(0, y_min - self.bbox_padding)
            x_max = min(W - 1, x_max_excl - 1 + self.bbox_padding)
            y_max = min(H - 1, y_max_excl - 1 + self.bbox_padding)
            boxes.append([x_min, y_min, x_max, y_max])
            component_masks.append((cc == region.label).astype(np.uint8))

        sample["mask"] = torch.from_numpy(mask).unsqueeze(0).to(torch.uint8)
        sample["component_masks"] = [torch.from_numpy(m).unsqueeze(0).to(torch.uint8) for m in component_masks]
        sample["bbox"] = torch.tensor(boxes, dtype=torch.float32) if boxes else torch.zeros((0, 4), dtype=torch.float32)
        return sample


class ResizeSampleMultiBox:
    """Resize image/mask and scale an [N,4] bbox tensor."""

    def __init__(self, size: tuple[int, int]):
        self.size = size

    def __call__(self, sample):
        image = sample["image"]
        _, old_h, old_w = image.shape
        new_h, new_w = self.size

        sample["image"] = F.interpolate(
            image.unsqueeze(0), size=self.size, mode="bilinear", align_corners=False
        ).squeeze(0)

        if sample.get("mask") is not None:
            sample["mask"] = F.interpolate(
                sample["mask"].float().unsqueeze(0), size=self.size, mode="nearest"
            ).squeeze(0).to(torch.uint8)

        if sample.get("component_masks") is not None:
            resized_components = []
            for component in sample["component_masks"]:
                resized = F.interpolate(
                    component.float().unsqueeze(0), size=self.size, mode="nearest"
                ).squeeze(0).to(torch.uint8)
                resized_components.append(resized)
            sample["component_masks"] = resized_components

        boxes = sample.get("bbox")
        if boxes is not None and len(boxes) > 0:
            boxes = boxes.float().clone()
            boxes[:, [0, 2]] *= new_w / old_w
            boxes[:, [1, 3]] *= new_h / old_h
            sample["bbox"] = boxes

        sample["original_size"] = (old_h, old_w)
        sample["resized_size"] = self.size
        return sample

In [ ]:
def mammo_collate_fn(batch):
    out = {
        "image": torch.stack([sample["image"] for sample in batch], dim=0),
        "label": torch.stack([sample["label"] for sample in batch], dim=0),
        "mask": torch.stack([sample["mask"] for sample in batch], dim=0),
        "bbox": [sample["bbox"] for sample in batch],
        "component_masks": [sample.get("component_masks", []) for sample in batch],
        "path": [sample["path"] for sample in batch],
        "annotation_path": [sample["annotation_path"] for sample in batch],
        "metadata": [sample.get("metadata", {}) for sample in batch],
        "original_size": [sample.get("original_size") for sample in batch],
    }
    return out


medsam_transform = ComposeSampleTransforms([
    ConnectedComponentBoxesFromMask(
        min_area=MIN_COMPONENT_AREA,
        bbox_padding=BBOX_PADDING,
        threshold=0.0,
    ),
    EnsureImageChannels(3),
    ResizeSampleMultiBox(MEDSAM_IMAGE_SIZE),
])

zgt_dataset = ZGTVersaMammoDataset(
    root=TEST_ROOT,
    image_suffixes=(".jpg", ".png"),
    annotation_format="mask_image",
    skip_empty_mask=False,
    return_metadata=True,
    transform=medsam_transform,
)

zgt_loader = DataLoader(
    zgt_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=mammo_collate_fn,
)

print(f"Loaded {len(zgt_dataset)} ZGT test images")

In [ ]:
def dice_score(pred_mask: torch.Tensor, gt_mask: torch.Tensor) -> float:
    pred = pred_mask.bool()
    gt = gt_mask.bool()
    denom = pred.sum() + gt.sum()
    if denom.item() == 0:
        return 1.0
    return float((2 * (pred & gt).sum() / denom).item())


def run_medsam_for_sample(model, image: torch.Tensor, boxes: torch.Tensor):
    # image: [3, 1024, 1024], boxes: [N,4] in 1024 coordinates.
    H, W = image.shape[-2:]
    if boxes is None or len(boxes) == 0:
        return torch.zeros((1, H, W), dtype=torch.uint8, device=image.device), []

    image_batch = image.unsqueeze(0).to(DEVICE)
    img_embed = model.image_encoder(image_batch)

    prompt_masks = []
    for box in boxes.to(DEVICE):
        pred = medsam_inference_single(
            medsam_model=model,
            img_embed=img_embed,
            box_1024=box,
            H=H,
            W=W,
            threshold=MASK_THRESHOLD,
        )
        prompt_masks.append(pred.squeeze(0).to(torch.uint8))  # [1,H,W]

    union = torch.zeros_like(prompt_masks[0])
    for mask in prompt_masks:
        union = torch.logical_or(union.bool(), mask.bool()).to(torch.uint8)
    return union, prompt_masks

In [ ]:
def to_display_image(image_tensor: torch.Tensor) -> np.ndarray:
    img = image_tensor.detach().cpu().permute(1, 2, 0).numpy()
    img = (img - img.min()) / max(img.max() - img.min(), 1e-8)
    return img


def overlay_result(image_tensor, gt_mask, pred_mask, boxes, title, out_path):
    image = to_display_image(image_tensor)
    gt = gt_mask.detach().cpu().squeeze().numpy().astype(bool)
    pred = pred_mask.detach().cpu().squeeze().numpy().astype(bool)

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(image, cmap="gray")

    gt_rgba = np.zeros((*gt.shape, 4), dtype=np.float32)
    gt_rgba[..., 1] = 1.0
    gt_rgba[..., 3] = gt * 0.35
    ax.imshow(gt_rgba)

    pred_rgba = np.zeros((*pred.shape, 4), dtype=np.float32)
    pred_rgba[..., 0] = 1.0
    pred_rgba[..., 3] = pred * 0.35
    ax.imshow(pred_rgba)

    for i, box in enumerate(boxes.detach().cpu().numpy() if boxes is not None else []):
        x1, y1, x2, y2 = box.tolist()
        rect = plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="yellow", linewidth=1.5)
        ax.add_patch(rect)
        ax.text(x1, max(0, y1 - 4), f"GT box {i+1}", color="black", fontsize=8, bbox={"facecolor": "yellow", "alpha": 0.8, "pad": 1})

    ax.set_title(title)
    ax.set_axis_off()
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, bbox_inches="tight", pad_inches=0, dpi=160)
    plt.close(fig)

In [ ]:
if not MEDSAM_CHECKPOINT.exists():
    raise FileNotFoundError(
        f"MedSAM checkpoint not found: {MEDSAM_CHECKPOINT}\n"
        "Update MEDSAM_CHECKPOINT in the configuration cell."
    )

print(f"Loading MedSAM checkpoint: {MEDSAM_CHECKPOINT}")
model = sam_model_registry["vit_b"](checkpoint=str(MEDSAM_CHECKPOINT)).to(DEVICE).eval()
print("MedSAM loaded")

In [ ]:
MASK_OUT_DIR.mkdir(parents=True, exist_ok=True)
OVERLAY_OUT_DIR.mkdir(parents=True, exist_ok=True)

rows = []

with torch.no_grad():
    for batch_idx, batch in enumerate(zgt_loader, start=1):
        images = batch["image"]
        gt_masks = batch["mask"]
        boxes_list = batch["bbox"]
        paths = batch["path"]

        for i in range(images.shape[0]):
            image = images[i].to(DEVICE)
            gt_mask = gt_masks[i].to(DEVICE)
            boxes = boxes_list[i]
            image_name = Path(paths[i]).parent.name

            pred_union, prompt_masks = run_medsam_for_sample(model, image, boxes)
            dice = dice_score(pred_union.cpu(), gt_mask.cpu())

            io.imsave(
                MASK_OUT_DIR / f"{image_name}_pred_union.png",
                (pred_union.detach().cpu().squeeze().numpy() * 255).astype(np.uint8),
                check_contrast=False,
            )
            for prompt_idx, prompt_mask in enumerate(prompt_masks, start=1):
                io.imsave(
                    MASK_OUT_DIR / f"{image_name}_prompt{prompt_idx}.png",
                    (prompt_mask.detach().cpu().squeeze().numpy() * 255).astype(np.uint8),
                    check_contrast=False,
                )

            overlay_result(
                image_tensor=image.cpu(),
                gt_mask=gt_mask.cpu(),
                pred_mask=pred_union.cpu(),
                boxes=boxes,
                title=f"{image_name} | Dice={dice:.3f} | GT component prompts={len(boxes)}",
                out_path=OVERLAY_OUT_DIR / f"{image_name}.png",
            )

            rows.append({
                "image_name": image_name,
                "dice": dice,
                "num_gt_component_boxes": int(len(boxes)),
                "gt_area": int(gt_mask.bool().sum().item()),
                "pred_area": int(pred_union.bool().sum().item()),
            })

        if batch_idx % 5 == 0 or batch_idx == len(zgt_loader):
            print(f"Processed batch {batch_idx}/{len(zgt_loader)}")

metrics_df = pd.DataFrame(rows)
metrics_df.to_csv(METRICS_OUT, index=False)

summary = {
    "dataset": DATASET_NAME,
    "num_images": int(len(metrics_df)),
    "mean_dice": float(metrics_df["dice"].mean()) if len(metrics_df) else 0.0,
    "median_dice": float(metrics_df["dice"].median()) if len(metrics_df) else 0.0,
    "std_dice": float(metrics_df["dice"].std(ddof=1)) if len(metrics_df) > 1 else 0.0,
    "min_component_area": MIN_COMPONENT_AREA,
    "bbox_padding": BBOX_PADDING,
    "mask_threshold": MASK_THRESHOLD,
    "medsam_image_size": MEDSAM_IMAGE_SIZE,
}
with open(SUMMARY_OUT, "w") as f:
    json.dump(summary, f, indent=2)

summary

In [ ]:
# Preview a few overlays.
preview_paths = sorted(OVERLAY_OUT_DIR.glob("*.png"))[:6]
if not preview_paths:
    print("No overlays generated yet.")
else:
    fig, axes = plt.subplots(len(preview_paths), 1, figsize=(9, 6 * len(preview_paths)))
    if len(preview_paths) == 1:
        axes = [axes]
    for ax, path in zip(axes, preview_paths):
        ax.imshow(io.imread(path))
        ax.set_title(path.name)
        ax.set_axis_off()
    plt.tight_layout()